# H100 음성·음악 후속 실험

기존 H100 `wsp-9683432r72m7`에서 실행합니다. 입력과 모델을 먼저 업로드하고 검증합니다. 시작 후 최대 60분이며, 마지막 10분은 결과 보존에 남깁니다. 완료 또는 오류 시 결과 ZIP의 모든 SHA-256을 확인한 뒤 GPU를 정지합니다.

음성은 이전에 열어 본 4명 화자의 진단 자료입니다. 새 음악은 FMA 64곡과 SONICS 38곡입니다. 독립 그룹 106개로, 독립 원천 300개 기준에 못 미칩니다. 음성 라벨 청취 검수와 가창 검증도 끝나지 않았습니다. 이 노트북은 원인 진단용이며 공식 점수나 배포 개선을 입증하지 않습니다. FMA/SONICS 출처 편향과 사전학습 중복 가능성도 남아 있습니다.

In [ ]:
from voice_music_h100_experiments import Experiments
run = Experiments('/root/voice_music_h100_20260917/config.json')

## 1. 같은 길이·음량의 음성과 혼합음 비교

HTDemucs는 파일당 한 번 실행하고 스템을 재사용합니다. raw와 vocals의 DF-Arena 출력을 비교합니다. 혼합된 실제 음악의 음성 라벨은 미확인 상태로 둡니다.

In [ ]:
run.experiment_1()

## 2. AntiDeepfake와 zero-shot 평균 비교

음악 M330, PANNs, File max를 고정합니다. 원본 Fairseq와 로짓 일치를 먼저 확인합니다. max와 top-30% pooling은 별도 후보로 기록합니다.

In [ ]:
run.experiment_2()

## 3. 음성 residual head 진단

기존 480개 음성의 화자 메타데이터를 복원해 분리하고 작은 head만 학습합니다. 가창 검증을 통과한 배포 모델로 취급하지 않습니다.

In [ ]:
run.experiment_3()

## 4. 새 음악 자료 추가 학습

NSynth를 두 학습군에서 모두 제외합니다. 나머지 기존 데이터와 새 데이터 추가군을 같은 seed 41·17 epochs로 비교합니다.

In [ ]:
run.experiment_4()

## 5. 독립 calibration과 File 결합

calibration 그룹으로만 Platt 계수를 학습합니다. dev로 max와 noisy-OR를 비교합니다. VOICE_FAKE와 MUSIC_FAKE 출력에는 존재 확률을 곱하지 않습니다.

In [ ]:
run.experiment_5()

## 6. 음질 회귀 검사와 조건부 seed 반복

dev의 기본·4초·MP3·전화 조건에서 File/Music EER가 모두 악화되지 않고 기본 점수가 개선될 때만 seed 42·43을 같은 17 epochs로 학습합니다. 동률이면 추가 계산을 중단합니다. 선택을 저장한 뒤 test를 보고합니다.

In [ ]:
run.experiment_6()